In [4]:
import pandas as pd
import glob
import os

# Set up the paths for the input and output
input_folder = './'  # Current folder where the script is
output_folder = '../../Processed Data'

# Make sure the output directory actually exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Get a list of all CSV files in the current directory
csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

# Mapping to separate the causes: 0=Normal, 1=Cooking, 2=Other
status_to_label = {
    0: 'Normal',
    1: 'Cooking',
    2: 'Fire'
}

# Mapping sensor names to my target variable names
rename_map = {
    'Temperature': 'temperature',
    'Humidity': 'humidity',
    'TVOC': 'tvoc_ppb',
    'eCO2': 'eco2_ppm'
}

# Start the loop to process each file automatically
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}...")
    
    # Load the data
    df = pd.read_csv(file_path)
    
    # Check if the 'Status' column exists before trying to map it
    if 'Status' not in df.columns:
        print(f"Skipping {file_name}: 'Status' column not found.")
        continue

    # Map labels and class IDs
    df['label'] = df['Status'].map(status_to_label)
    df['class_id'] = df['Status']

    # --- THE DYNAMIC COLUMN FINDER ---
    # We look for any column name that contains our target word (case-insensitive)
    # The 'next()' function grabs the first match it finds. 
    # If it finds nothing, it returns None.
    actual_temp = next((col for col in df.columns if 'temp' in col.lower()), None)
    actual_hum = next((col for col in df.columns if 'humid' in col.lower()), None)
    actual_tvoc = next((col for col in df.columns if 'tvoc' in col.lower()), None)
    actual_eco2 = next((col for col in df.columns if 'eco2' in col.lower() or 'co2' in col.lower()), None)

    # Check if we successfully found all 4 required columns
    if not all([actual_temp, actual_hum, actual_tvoc, actual_eco2]):
        print(f"Skipping {file_name}: Could not find all 4 sensor columns.")
        continue

    # Build a custom rename map for THIS specific file based on what we found
    dynamic_rename_map = {
        actual_temp: 'temperature',
        actual_hum: 'humidity',
        actual_tvoc: 'tvoc_ppb',
        actual_eco2: 'eco2_ppm'
    }

    # Filter and rename using our dynamic map
    final_df = df[['label', 'class_id', actual_temp, actual_hum, actual_tvoc, actual_eco2]].rename(columns=dynamic_rename_map)
    
    # Convert sensor data to numeric and drop bad rows
    cols_to_fix = ['temperature', 'humidity', 'tvoc_ppb', 'eco2_ppm']
    final_df[cols_to_fix] = final_df[cols_to_fix].apply(pd.to_numeric, errors='coerce')
    final_df = final_df.dropna()
    
    # Define the output path and save
    save_path = os.path.join(output_folder, f"UAE_cleaned_{file_name}")
    final_df.to_csv(save_path, index=False)

print("\nAll files have been processed and saved to the 'Processed Data' folder!")

Processing: carton_1.csv...
Processing: carton_2.csv...
Processing: clothing_1.csv...
Processing: clothing_2.csv...
Processing: electrical_1.csv...
Processing: electrical_2.csv...
Processing: electrical_3.csv...
Processing: electrical_4.csv...

All files have been processed and saved to the 'Processed Data' folder!
